In [1]:
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

# Suppress specific warnings (e.g., UserWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Suppress warnings from a specific module
warnings.filterwarnings("ignore", module="torch")

In [2]:
from dataloaderlite import DataLoaderLite
from transformers import AutoTokenizer
import torch

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda:1'
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"using device: {device}")

# SEED
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

# STOP
num_return_sequences = 5
max_length = 30

train_loader = DataLoaderLite(B = 32, T = 1024)
# Define the fixed text for prediction
fixed_text = "This is a fixed text used for prediction."
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/cosmo2-tokenizer")
encoded_text = tokenizer(fixed_text, return_tensors="pt").to(device)
print (f"Encoded Text {encoded_text}")

using device: cuda:1
Streaming dataset loaded.
Encoded Text {'input_ids': tensor([[ 1348,   314,   253,  6450,  1694,   804,   327, 12435,    30]],
       device='cuda:1'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:1')}


In [3]:
# Use
from deepseek_smollm2 import DeepSeekModel, DeepSeekConfig
model = DeepSeekModel(DeepSeekConfig())

# Wrap model in DataParallel
#if torch.cuda.device_count() > 1:
#    print(f"Using {torch.cuda.device_count()} GPUs!")
#    model = torch.nn.DataParallel(model)  # Enable Multi-GPU

model.to(device)
model = torch.compile(model)

# NEW CODE
import time
optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-4)

In [4]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.capture_scalar_outputs = True
torch._dynamo.config.assume_static_by_default = True

import os
os.environ["TORCH_LOGS"] = "+dynamo"
os.environ["TORCHDYNAMO_VERBOSE"] = "1"

In [5]:
# Training loop
import time
total_steps = 10000
prediction_interval = 250
checkpoint_path = "llm_checkpoint.pt"
model_path = "model.pt"

for step in range(total_steps):
    t0 = time.time()
    x, y = train_loader.next_batch()
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    # NEW CODE ADDED HERE
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        logits, loss = model(x, y) 
    loss.backward()
    optimizer.step()
    torch.cuda.synchronize() 
    t1 = time.time()
    dt = (t1 - t0) * 1000
    tokens_per_sec = (train_loader.B * train_loader.T) / (t1 - t0)
    print(f'Step: {step} | loss: {loss.item():.4f} | dt: {dt:.2f}ms | tok/sec: {tokens_per_sec: .2f}')
    
    # Perform prediction every 500 steps
    if (step + 1) % prediction_interval == 0:
        with torch.no_grad():
            model.eval()
            
            max_new_tokens = 50
            temperature = 1.0
            top_k = 40
            
            prediction_logits = model.generate(encoded_text["input_ids"], max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
            prediction = tokenizer.decode(prediction_logits[0], skip_special_tokens=True)
            print(f"Prediction at step {step+1}: \n : {prediction} \n")
            model.train()

# Save checkpoint after 10000 steps
torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict()}, checkpoint_path)
print(f"Checkpoint saved to {checkpoint_path}")
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

W0316 02:51:28.456000 598206 torch/fx/experimental/symbolic_shapes.py:6307] [0/0] failed during evaluate_expr(zuf0, hint=None, size_oblivious=False, forcing_spec=False
E0316 02:51:28.458000 598206 torch/fx/experimental/recording.py:299] [0/0] failed while running evaluate_expr(*(zuf0, None), **{'fx_node': False})
W0316 02:51:30.217000 598206 torch/fx/experimental/symbolic_shapes.py:6307] [1/1] failed during evaluate_expr(zuf0, hint=None, size_oblivious=False, forcing_spec=False
E0316 02:51:30.220000 598206 torch/fx/experimental/recording.py:299] [1/1] failed while running evaluate_expr(*(zuf0, None), **{'fx_node': False})
W0316 02:51:30.611000 598206 torch/fx/experimental/symbolic_shapes.py:6307] [2/0] failed during evaluate_expr(zuf0, hint=None, size_oblivious=False, forcing_spec=False
E0316 02:51:30.613000 598206 torch/fx/experimental/recording.py:299] [2/0] failed while running evaluate_expr(*(zuf0, None), **{'fx_node': False})


Step: 0 | loss: 15.3196 | dt: 32993.58ms | tok/sec:  993.16
Step: 1 | loss: 13.3045 | dt: 528.00ms | tok/sec:  62060.62
Step: 2 | loss: 12.2670 | dt: 526.34ms | tok/sec:  62255.88
Step: 3 | loss: 11.9211 | dt: 527.92ms | tok/sec:  62070.07
Step: 4 | loss: 11.4376 | dt: 531.68ms | tok/sec:  61631.35
Step: 5 | loss: 11.1130 | dt: 528.65ms | tok/sec:  61984.66
Step: 6 | loss: 10.6499 | dt: 553.06ms | tok/sec:  59248.54
Step: 7 | loss: 10.3994 | dt: 524.05ms | tok/sec:  62527.99
Step: 8 | loss: 10.3517 | dt: 536.72ms | tok/sec:  61052.26
Step: 9 | loss: 10.1276 | dt: 556.19ms | tok/sec:  58914.89
Step: 10 | loss: 9.8995 | dt: 528.03ms | tok/sec:  62056.95
Step: 11 | loss: 9.6302 | dt: 523.06ms | tok/sec:  62647.15
Step: 12 | loss: 9.3733 | dt: 535.12ms | tok/sec:  61235.19
Step: 13 | loss: 9.2752 | dt: 554.02ms | tok/sec:  59146.17
Step: 14 | loss: 9.1447 | dt: 532.76ms | tok/sec:  61506.52
Step: 15 | loss: 8.9205 | dt: 528.10ms | tok/sec:  62048.96
Step: 16 | loss: 8.7197 | dt: 549.76ms |

In [6]:
import torch
for i in range(5):
    with torch.no_grad():
        model.eval()
        
        max_new_tokens = 10
        temperature = 1.0
        top_k = 40
        
        prediction_logits = model.generate(encoded_text["input_ids"], max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
        prediction = tokenizer.decode(prediction_logits[0], skip_special_tokens=True)
        print(f"Prediction output {i+1}: \n : {prediction} \n")
        model.train()

Prediction output 1: 
 : This is a fixed text used for prediction. With a critical role in the end of the story 

Prediction output 2: 
 : This is a fixed text used for prediction.
B. Moreover, or wrong. And let 

Prediction output 3: 
 : This is a fixed text used for prediction. Each time on the `get of its core components 

Prediction output 4: 
 : This is a fixed text used for prediction. It's dive right in the importance of the best 

Prediction output 5: 
 : This is a fixed text used for prediction. The Fascinating Topics covered by the process where people can 



In [7]:
# Load checkpoint and train for another 500 steps
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

print("Loaded checkpoint. Resuming training for another 50 steps.")

for step in range(500):
    t0 = time.time()

    # Training batch
    x, y = train_loader.next_batch()
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        logits, loss = model(x, y)
    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()

    t1 = time.time()
    dt = (t1 - t0) * 1000  # Time in milliseconds
    tokens_per_sec = (train_loader.B * train_loader.T) / (t1 - t0)

    print(f"Step {step+1} (After Resuming) | Loss: {loss.item():.4f} | dt: {dt:.2f}ms | tok/sec: {tokens_per_sec:.2f}")
    
print(loss)

Loaded checkpoint. Resuming training for another 50 steps.
Step 1 (After Resuming) | Loss: 4.8454 | dt: 753.94ms | tok/sec: 43462.35
Step 2 (After Resuming) | Loss: 4.8272 | dt: 542.21ms | tok/sec: 60434.59
Step 3 (After Resuming) | Loss: 4.9117 | dt: 535.26ms | tok/sec: 61218.71
Step 4 (After Resuming) | Loss: 4.9445 | dt: 549.35ms | tok/sec: 59648.83
Step 5 (After Resuming) | Loss: 4.9309 | dt: 538.59ms | tok/sec: 60840.11
Step 6 (After Resuming) | Loss: 4.9266 | dt: 540.43ms | tok/sec: 60633.67
Step 7 (After Resuming) | Loss: 4.8277 | dt: 531.64ms | tok/sec: 61636.10
Step 8 (After Resuming) | Loss: 4.8534 | dt: 523.34ms | tok/sec: 62613.59
Step 9 (After Resuming) | Loss: 4.8304 | dt: 548.11ms | tok/sec: 59783.70
Step 10 (After Resuming) | Loss: 4.8726 | dt: 533.30ms | tok/sec: 61444.32
Step 11 (After Resuming) | Loss: 4.9027 | dt: 526.40ms | tok/sec: 62249.12
Step 12 (After Resuming) | Loss: 4.8465 | dt: 535.51ms | tok/sec: 61189.96
Step 13 (After Resuming) | Loss: 4.8783 | dt: 531.